# Benchmarking

In [15]:
import numpy as np
import jax
import jax.numpy as jnp
import time
import WDM

We will time some of the dwt methods and their inverses on a time series of length $N=2^{14}=16384$. We will also show that they agree exactly. 

In [16]:
wdm = WDM.WDM.WDM_transform(dt=1., 
                            Nf=2**8, 
                            N=2**14)

x = np.random.normal(size=wdm.N) # white noise

In [17]:
t0 = time.time()
w = wdm.forward_transform_exact(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 3184.45 milliseconds


In [18]:
t0 = time.time()
x_recovered = wdm.inverse_transform_exact(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

max_error = jnp.max(jnp.abs(x - x_recovered))

relative_error = (
    jnp.linalg.norm(x - x_recovered)
    / jnp.linalg.norm(x)
)
print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Time taken: 9868.36 milliseconds
Maximum error:  9.518e-12
Relative error: 1.484e-12


In [19]:
t0 = time.time()
w = wdm.forward_transform_short_fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 220.21 milliseconds


The short FFT method, has no inverse call. 

The forward_transform_fft method is the one used in production when wdm.dwt is called. 

In [20]:
t0 = time.time()
w = wdm.forward_transform_fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 94.53 milliseconds


In [21]:
t0 = time.time()
x_recovered = wdm.inverse_transform_fft(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

max_error = jnp.max(jnp.abs(x - x_recovered))

relative_error = (
    jnp.linalg.norm(x - x_recovered)
    / jnp.linalg.norm(x)
)
print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Time taken: 108.98 milliseconds
Maximum error:  2.665e-15
Relative error: 6.384e-16


Compilation means subsequent calls can be much faster.

In [22]:
t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 5.72 milliseconds
Time taken: 1.33 milliseconds


Same thing holds for an inverse transform:

In [23]:
t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 2.91 milliseconds
Time taken: 1.02 milliseconds


Vectorisation means that batched transforms can also be much faster.

In [24]:
x = np.random.normal(size=(20, wdm.N)) # white noise

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = wdm.dwt(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 67.69 milliseconds
Time taken: 5.56 milliseconds


As above, same thing holds for the inverse transform:

In [25]:
t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
x = wdm.idwt(w).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 128.33 milliseconds
Time taken: 8.10 milliseconds


Let's compare this with the cost of an FFT on the same set of time series.

In [26]:
t0 = time.time()
w = jnp.fft.fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

t0 = time.time()
w = jnp.fft.fft(x).block_until_ready()
t1 = time.time()

print(f"Time taken: {(t1-t0)*1000:.2f} milliseconds")

Time taken: 8.63 milliseconds
Time taken: 6.51 milliseconds


The new inverse and forward transform is also independent of $q$ as a paremeter.

In [28]:
x = np.random.normal(size=wdm.N) # white noise

q_vals = [2, 4, 8, 16]

for q in q_vals:
    wdm_q = WDM.WDM.WDM_transform(dt=1., 
                            Nf=2**8, 
                            N=2**14,
                            q = q,)

    w_q = wdm_q.forward_transform_fft(x)

    # Compile first
    wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print(f"q = {q}")

    print("fast FFT inverse:")
    %timeit wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print()

q = 2
fast FFT inverse:
378 μs ± 11.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 4
fast FFT inverse:
376 μs ± 10.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 8
fast FFT inverse:
380 μs ± 4.02 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

q = 16
fast FFT inverse:
366 μs ± 21 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

